# Metrics and Evaluation - California Housing Dataset

In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

x, y = fetch_california_housing(return_X_y=True,as_frame=True)

x.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [2]:


x_train, x_testval, y_train, y_testval = train_test_split(x, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_testval, y_testval, test_size=0.5, random_state=42)

In [3]:
cols = set(x.columns) - {"Longitude", "Latitude"}
Q1 = x_train.quantile(0.25)
Q3 = x_train.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

In [4]:
x_train_tensorial = torch.tensor(x_train.values, dtype=torch.float32)
y_train_tensorial = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

x_val_tensorial = torch.tensor(x_val.values, dtype=torch.float32)
y_val_tensorial = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)

x_test_tensorial = torch.tensor(x_test.values, dtype=torch.float32)
y_test_tensorial = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

In [5]:
mean = torch.mean(x_train_tensorial, dim=0)
std = torch.std(x_train_tensorial, dim=0)
std[std == 0] = 2.220446049250313e-16 # no me va a pasar DOS veces, por eso el epsilon

In [6]:
class modelo_hotel_california(nn.Module):
    def __init__(self, entrada, n_capas):
        super().__init__()
        
        self.mean = mean
        self.std = std
        self.lower_bound = torch.tensor(lower_bound.values, dtype=torch.float32)
        self.upper_bound = torch.tensor(upper_bound.values, dtype=torch.float32)
        self.columns = x_train.columns

        self.red_californiana = nn.Sequential(
            nn.Linear(entrada, n_capas),
            nn.ReLU(),
            nn.Linear(n_capas, 1) # NO ERA SOFTMAX??
        )

    def forward(self, x):
        for i in range(x.shape[1]):
            if self.columns[i] != "Latitude" and self.columns[i] != "Longitude":
                x[:,i] = torch.clamp(x[:,i], min=self.lower_bound[i], max=self.upper_bound[i])
                x[:,i] = (x[:,i] - self.mean[i]) / self.std[i]
        return self.red_californiana(x)
   

In [7]:
def entrenar(x_train_tensorial, y_train_tensorial, x_val_tensorial, y_val_tensorial, n_capas, lr, epoches):
    modelo_californiano = modelo_hotel_california(x_train_tensorial.shape[1], n_capas)

    sandler_californiano = optim.Adam(modelo_californiano.parameters(), lr = lr)
    loss_fn = nn.MSELoss()

    for _ in range(epoches):

        sandler_californiano.zero_grad()
        # LO CAMBIA TODO
        loss = loss_fn(modelo_californiano(x_train_tensorial), y_train_tensorial)
        loss.backward()
        
        sandler_californiano.step()
        
        # ya me lo aprendi

    with torch.no_grad():
        val_loss = loss_fn(modelo_californiano(x_val_tensorial), y_val_tensorial).item()
        
    return modelo_californiano, val_loss
   

In [8]:
resultados = []

for configuracion_californiana in [(64, 0.001, 120), (64, 0.001, 240), (64, 0.001, 320), (64, 0.001, 50), (64, 0.001, 12),(32, 0.001, 120), (32, 0.001, 240), (32, 0.001, 320), (32, 0.001, 50), (32, 0.001, 12),(128, 0.001, 120), (128, 0.001, 240), (128, 0.001, 320), (128, 0.001, 50), (128, 0.001, 12),(320, 0.001, 120), (320, 0.001, 240), (320, 0.001, 320), (320, 0.001, 50), (320, 0.001, 12),(80, 0.001, 120), (80, 0.001, 240), (80, 0.001, 320), (80, 0.001, 50), (80, 0.001, 12),(500, 0.0001, 120), (64, 0.01, 120),  (64, 0.01, 240),  (64, 0.01, 320),  (64, 0.01, 50),  (64, 0.01, 12),(32, 0.01, 120),(32, 0.01, 240),(32, 0.01, 320),(32, 0.01, 50),(32, 0.01, 12),(128, 0.01, 120), (128, 0.01, 240), (128, 0.01, 320), (128, 0.01, 50), (128, 0.01, 12),(320, 0.01, 120), (320, 0.01, 240), (320, 0.01, 320), (320, 0.01, 50), (320, 0.01, 12),(80, 0.01, 120),(80, 0.01, 240),(80, 0.01, 320),(80, 0.01, 50),(80, 0.01, 12),(500, 0.0001, 120)]: #esto esta mas limpio que ponerle 3000 celdas con MD apoco no profe
    modelo_californiano, val_loss = entrenar(
        x_train_tensorial, y_train_tensorial,
        x_val_tensorial, y_val_tensorial,
        configuracion_californiana[0],
        configuracion_californiana[1],
        configuracion_californiana[2]
    )

    resultados.append((modelo_californiano, val_loss, configuracion_californiana))
    print(configuracion_californiana, val_loss)

(64, 0.001, 120) 1.3175618648529053
(64, 0.001, 240) 1.450573444366455
(64, 0.001, 320) 1.2770905494689941
(64, 0.001, 50) 1.3124232292175293
(64, 0.001, 12) 2.8669307231903076
(32, 0.001, 120) 1.2799386978149414
(32, 0.001, 240) 1.2475777864456177
(32, 0.001, 320) 1.2627098560333252
(32, 0.001, 50) 4.261107921600342
(32, 0.001, 12) 112.43585205078125
(128, 0.001, 120) 1.2446362972259521
(128, 0.001, 240) 1.2456793785095215
(128, 0.001, 320) 1.2739261388778687
(128, 0.001, 50) 1.285675048828125
(128, 0.001, 12) 17.369813919067383
(320, 0.001, 120) 1.2790443897247314
(320, 0.001, 240) 1.2352489233016968
(320, 0.001, 320) 1.285267949104309
(320, 0.001, 50) 1.3710914850234985
(320, 0.001, 12) 9.704707145690918
(80, 0.001, 120) 1.2882013320922852
(80, 0.001, 240) 1.2714391946792603
(80, 0.001, 320) 1.3095457553863525
(80, 0.001, 50) 1.2960739135742188
(80, 0.001, 12) 3.2085278034210205
(500, 0.0001, 120) 1.2353674173355103
(64, 0.01, 120) 1.274776816368103
(64, 0.01, 240) 1.252994894981384

In [9]:
california_si_no_especulara, _, configuracion_californiana = min(resultados, key=lambda x: x[1])

print("Mejor modelo (capas, learning rate)", configuracion_californiana)

Mejor modelo (capas, learning rate) (320, 0.001, 240)


In [10]:
with torch.no_grad():
    test_loss = nn.MSELoss()(california_si_no_especulara(x_test_tensorial), y_test_tensorial).item()

print("Evaluación", test_loss)

Evaluación 1.3210045099258423


# Evaluación y comentarios finales.
Se eligió una arquitectura relativamente sencilla para nada más modelar una regresión y poder variar muchos hiperparámetros con el fin de demostrar su efectividad en la práctica. Bien, este, se limita por esto mismo, variando, así que variando el número de neuronas y épocas impacta el resultado, pero no de forma dramática, lo que podría decirnos que hace falta algo de ajuste fino, que bueno, pues, no venía requerido, pero podría haberse implementado. Los casos donde el loss se dispara, por ejemplo, con muy pocas épocas, reflejan el entrenamiento siendo insuficiente, pero, pues, no un problema de la arquitectura per se. ya que estamos evaluando el mean square error, los valores difieren por muy poquito, ¿no? Sí, o sea, tenemos números de o sea, variaciones de MSE en las unidades, lo que a mi parecer, sí, ya sé que el el dataset está en cantidades grandotas, ¿no? Las unidades no son unidades de dólar. pero de todas formas, o sea, tenemos un error que, pues, digo, es aceptable, opino yo. Y el mejor modelo terminó siendo el de 320 capas, con un learning rate de 0.001 para no dar pasos agigantados, y 240 épocas. La evaluación le dio un MSE de 1.32, lo que, pues, volvemos a lo mismo, no es excesivo. Digo, comparado, por ejemplo, el de 112, solo digo.